# 06 - Hệ thống Gợi ý Bài tập
## Complete Recommendation System Demo

Mục tiêu:
- Kết hợp tất cả kết quả: TF-IDF + GMM + NN + AR
- Demo `recommend_exercises()` với nhiều ví dụ
- Trực quan hóa pipeline

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

plt.style.use('seaborn-v0_8-darkgrid')
print("Ready!")

## 6.1 Pipeline Overview

In [ ]:
# Visualize pipeline
fig, ax = plt.subplots(figsize=(14, 4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4); ax.axis('off')

steps = [
    (0.5, 'Input Video\nText', '#1565c0'),
    (2.5, 'Text\nPreprocessing', '#2e7d32'),
    (4.5, 'TF-IDF\nVectors', '#f57f17'),
    (6.5, 'EM Clustering\n(GMM)', '#6a1b9a'),
    (8.5, 'Neural Network\n(Difficulty)', '#c62828'),
    (10.5, 'Association\nRules', '#00838f'),
    (12.5, 'Final\nRecommendation', '#37474f')
]

for x, label, color in steps:
    rect = mpatches.FancyBboxPatch((x-0.8, 1.2), 1.6, 1.6,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='white', alpha=0.9)
    ax.add_patch(rect)
    ax.text(x, 2.0, label, ha='center', va='center', fontsize=8,
            color='white', fontweight='bold')

# Arrows
for i in range(len(steps)-1):
    x1 = steps[i][0] + 0.8
    x2 = steps[i+1][0] - 0.8
    ax.annotate('', xy=(x2, 2.0), xytext=(x1, 2.0),
                arrowprops=dict(arrowstyle='->', color='#90a4ae', lw=2))

ax.set_title('Recommendation System Pipeline', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../report/images/nb06_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 6.2 Sử dụng recommend_exercises()

In [ ]:
from src.recommender import recommend_exercises

# Test 1: Toán học
result1 = recommend_exercises(
    "Introduction to Algebra: Solving Linear Equations",
    top_n=5,
    verbose=True
)

In [ ]:
# Test 2: Hóa học
result2 = recommend_exercises(
    "Organic Chemistry: Reaction Mechanisms and Functional Groups",
    top_n=5,
    verbose=True
)

In [ ]:
# Test 3: Lập trình
result3 = recommend_exercises(
    "Machine Learning with Python: Building Neural Networks from Scratch",
    top_n=5,
    verbose=True
)

In [ ]:
# Test 4: Lịch sử
result4 = recommend_exercises(
    "World War II: Causes, Key Events and Long-term Consequences",
    top_n=5,
    verbose=True
)

In [ ]:
# Test 5: Kinh tế
result5 = recommend_exercises(
    "Microeconomics: Supply and Demand Analysis",
    top_n=5,
    verbose=True
)

## 6.3 Biểu đồ kết quả gợi ý

In [ ]:
all_results = [result1, result2, result3, result4, result5]
queries = [
    "Algebra: Linear Equations",
    "Organic Chemistry",
    "Machine Learning / Neural Networks",
    "World War II",
    "Microeconomics"
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Recommendation System Results', fontsize=16, fontweight='bold')

# Topics detected
topics = [r['topic'] for r in all_results]
topic_counts = pd.Series(topics).value_counts()
axes[0].bar(topic_counts.index, topic_counts.values, color=plt.cm.viridis(np.linspace(0,1,len(topic_counts))))
axes[0].set_title('Topics Detected')
axes[0].set_ylabel('Count')

# Difficulties predicted
diffs = [r['difficulty'] for r in all_results]
diff_counts = pd.Series(diffs).value_counts()
colors_d = {'Easy': '#4caf50', 'Medium': '#ff9800', 'Hard': '#f44336'}
axes[1].bar(diff_counts.index, diff_counts.values, 
            color=[colors_d.get(d, '#90a4ae') for d in diff_counts.index])
axes[1].set_title('Difficulties Predicted')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../report/images/nb06_recommendation_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 6.4 Export kết quả

In [ ]:
# Export sample recommendations
export_rows = []
for q, r in zip(queries, all_results):
    for ex in r['exercises']:
        export_rows.append({
            'query': q,
            'topic': r['topic'],
            'difficulty': r['difficulty'],
            'cluster': r['cluster'],
            'rank': ex['rank'],
            'exercise': ex['exercise'],
            'score': ex['score']
        })

export_df = pd.DataFrame(export_rows)
export_df.to_csv('../data/processed/sample_recommendations.csv', index=False)
print(f"Exported {len(export_df)} recommendations")
export_df.head(15)

## 6.5 Summary

In [ ]:
print("="*60)
print("RECOMMENDATION SYSTEM - SUMMARY")
print("="*60)
print(f"\nPipeline components:")
print(f"  1. TF-IDF Vectorizer: 5000 features, bigrams")
print(f"  2. TruncatedSVD: 100 components")
print(f"  3. GaussianMixture: optimal k from BIC")
print(f"  4. Neural Network: MLP + DeepNN")
print(f"  5. Association Rules: FP-Growth")
print(f"\nTest results:")
for q, r in zip(queries, all_results):
    n_ex = r['metadata']['n_recommendations']
    print(f"  [{r['topic']:12s}] {r['difficulty']:6s} | Cluster {r['cluster']} | {n_ex} exercises")
print(f"\nStreamlit demo: streamlit run ../app.py")